In [1]:
import json
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats

### Setup (Loading data, minor preprocesing)

In [2]:
with open('dataset.json', 'r') as f:
    raw = json.load(f)

RESOLUTIONS = ['50', '71', '87', '100']
PARAMS = ['alpha_weight', 'hist_percent', 'filter_size', 'num_samples']

records = []

for scene, scene_data in raw.items():
    if not isinstance(scene_data, dict):
        continue
    for res in RESOLUTIONS:
        if res not in scene_data:
            continue
        for ref, ref_data in scene_data[res].items():
            if not ref.startswith('ref-'):
                continue
            if ref != f'ref-{scene}':
                continue
            for param in PARAMS:
                if param not in ref_data:
                    continue
                for entry in ref_data[param]:
                    records.append({
                        'scene': scene,
                        'resolution': int(res),
                        'parameter': param,
                        'value': entry['value'],
                        'score': entry['score'],
                    })

df = pd.DataFrame(records)

# Find common values per (parameter, resolution) across all scenes
common_values = df.groupby(['parameter', 'resolution', 'value'])['scene'].nunique()
max_scenes = df.groupby(['parameter', 'resolution'])['scene'].nunique()

def get_common_values(group):
    key = (group.name[1], group.name[2])  # (parameter, resolution)
    total_scenes = df[(df['parameter'] == key[0]) & (df['resolution'] == key[1])]['scene'].nunique()
    common = group.groupby('value')['scene'].nunique()
    return common[common == total_scenes].index.tolist()

# Center only over common values
def center_score(group):
    param, res = group['parameter'].iloc[0], group['resolution'].iloc[0]
    common = df[(df['parameter'] == param) & (df['resolution'] == res)]
    common_vals = common.groupby('value')['scene'].nunique()
    n_scenes = common['scene'].nunique()
    shared_vals = common_vals[common_vals == n_scenes].index
    mask = group['value'].isin(shared_vals)
    mean = group.loc[mask, 'score'].mean()
    return group['score'] - mean

df['score_centered'] = df.groupby(['scene', 'parameter', 'resolution'], group_keys=False).apply(center_score)
# Ignore hist percent less than 100 (measurement exists from prev trials but is not significant)
df = df[~((df['parameter'] == 'hist_percent') & (df['value'] < 100))]

# These are the weird outlier scenes--we cut them
EXCLUDE_SCENES = ['junkyard-mound1', 'junkyard-mound2', 'oldmine-speed-18', 'oldmine-speed-35', 'oldmine-speed-75', 'oldmine-speed-9', 'oldmine-warm']
df = df[~df['scene'].isin(EXCLUDE_SCENES)]

/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_47257/1545994299.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['score_centered'] = df.groupby(['scene', 'parameter', 'resolution'], group_keys=False).apply(center_score)


In [3]:
df.groupby(['parameter', 'resolution'])['score'].agg(['count', 'mean', 'std']).round(3)

count    mean    std
parameter    resolution                      
alpha_weight 50            252  86.141  8.881
             71            252  91.326  5.985
             87            252  93.199  4.957
             100           252  94.277  4.443
filter_size  50            146  87.444  7.482
             71            147  91.156  6.175
             87            147  92.444  5.636
             100           146  93.339  5.279
hist_percent 50             84  87.392  7.391
             71             84  91.853  5.674
             87             84  93.219  5.144
             100            82  94.015  4.814
num_samples  50            105  87.281  7.673
             71            105  91.172  6.186
             87            105  92.459  5.639
             100           105  93.279  5.299

### Basic Info about Data

In [4]:
print(f"Scenes: {df['scene'].nunique()}")
print(f"Scene names: {df['scene'].unique().tolist()}")
print(f"\nParameter value counts:")
print(df.groupby(['parameter', 'value'])['scene'].count().rename('n_scenes'))

Scenes: 21
Scene names: ['abandoned', 'abandoned-demo', 'abandoned-flipped', 'cubetest', 'fantasticvillage-open', 'lightfoliage', 'lightfoliage-close', 'oldmine', 'quarry-all', 'quarry-rocksonly', 'resto-close', 'resto-fwd', 'resto-pan', 'scifi', 'subway-lookdown', 'subway-turn', 'wildwest-bar', 'wildwest-barzoom', 'wildwest-behindcounter', 'wildwest-store', 'wildwest-town']

Parameter value counts:
parameter     value 
alpha_weight  0.01      84
              0.02      84
              0.04      84
              0.06      84
              0.10      84
              0.20      84
              0.50      84
              0.60      84
              0.70      84
              0.80      84
              0.90      84
              1.00      84
filter_size   0.10      84
              0.25      84
              0.50      83
              0.70      83
              0.90      84
              0.95      84
              1.00      84
hist_percent  100.00    84
              125.00    84
         

In [ ]:
# Print out for each scene x resolution, whether the param is calculated 
for param in PARAMS:
    sub = df[df['parameter'] == param]
    pivot = sub.groupby(['scene', 'resolution', 'value'])['score'].count().unstack('value')
    print(f"\n=== {param} ===")
    print(pivot.to_string())

In [ ]:
# Find missing params
for param in PARAMS:
    sub = df[df['parameter'] == param]
    pivot = sub.groupby(['scene', 'resolution', 'value'])['score'].count().unstack('value')
    missing = pivot[pivot.isna().any(axis=1)]
    if not missing.empty:
        print(f"\n=== {param} ===")
        print(missing.to_string())

### Plot of Parameter values x Resolution (both mean centered and not)

In [5]:
# Setup plot color & orders (for consistency)
res_order = ['100%', '87%', '71%', '50%']
res_colors = [ '#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']

color_scale = alt.Scale(domain=res_order, range=res_colors)

In [6]:
agg = df.groupby(['parameter', 'resolution', 'value'])['score'].agg(['mean', 'std', 'count']).reset_index()
agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
agg['resolution'] = agg['resolution'].astype(str) + '%'

y_min = (agg['mean'] - agg['ci95']).min() - 2
y_max = 100
y_scale = alt.Scale(domain=[y_min, y_max])

res_order = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

charts = []
for param in PARAMS:
    sub = agg[agg['parameter'] == param]
    band = alt.Chart(sub).mark_errorband(opacity=0.2).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', scale=y_scale),
        yError='ci95:Q',
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Mean CGVQM Score', scale=y_scale),
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order),
        tooltip=['resolution', 'value', alt.Tooltip('mean:Q', format='.3f'), alt.Tooltip('ci95:Q', format='.3f')]
    )
    charts.append((band + line).properties(title=f'{param}', width=280, height=200))

alt.vconcat(*[alt.hconcat(*charts[:2]), alt.hconcat(*charts[2:])]).resolve_scale(color='shared').properties(
    title='TAA Parameter Response Curves (Raw CGVQM)'
)

alt.VConcatChart(...)

In [7]:
agg = df.groupby(['parameter', 'resolution', 'value'])['score_centered'].agg(['mean', 'std', 'count']).reset_index()
agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
agg['resolution'] = agg['resolution'].astype(str) + '%'

y_min = (agg['mean'] - agg['ci95']).min() - 0.5
y_max = (agg['mean'] + agg['ci95']).max() + 0.5
y_scale = alt.Scale(domain=[y_min, y_max])

charts = []
for param in PARAMS:
    sub = agg[agg['parameter'] == param]
    band = alt.Chart(sub).mark_errorband(opacity=0.2).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', scale=y_scale),
        yError='ci95:Q',
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Mean Centered CGVQM', scale=y_scale),
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order),
        tooltip=['resolution', 'value', alt.Tooltip('mean:Q', format='.3f'), alt.Tooltip('ci95:Q', format='.3f')]
    )
    charts.append((band + line).properties(title=f'{param}', width=280, height=200))

alt.vconcat(*[alt.hconcat(*charts[:2]), alt.hconcat(*charts[2:])]).resolve_scale(color='shared').properties(
    title='TAA Parameter Response Curves (Mean-Centered CGVQM)'
)

alt.VConcatChart(...)

### Comparing how Alpha Weight Individual Scene shapes compare to the mean curve

In [16]:
alpha_df = df[df['parameter'] == 'alpha_weight'].copy()
alpha_df['resolution'] = alpha_df['resolution'].astype(str) + '%'

# Per-scene mean score per value
scene_curves = alpha_df.groupby(['scene', 'resolution', 'value'])['score_centered'].mean().reset_index()

# Mean curve across scenes
mean_curve = scene_curves.groupby(['resolution', 'value'])['score_centered'].agg(['mean','std','count']).reset_index()
mean_curve['ci95'] = 1.96 * mean_curve['std'] / np.sqrt(mean_curve['count'])

# Per-scene peak value
peaks = scene_curves.loc[scene_curves.groupby(['scene','resolution'])['score_centered'].idxmax()]
peaks = peaks.rename(columns={'value': 'peak_value'})

charts = []
for res in res_order:
    sc = scene_curves[scene_curves['resolution'] == res]
    mc = mean_curve[mean_curve['resolution'] == res]
    pk = peaks[peaks['resolution'] == res]
    col = res_colors[res_order.index(res)]

    # Individual scene lines (thin, transparent)
    individual = alt.Chart(sc).mark_line(opacity=0.15, strokeWidth=1).encode(
        x=alt.X('value:Q', title='alpha_weight'),
        y=alt.Y('score_centered:Q', title='Centered CGVQM', scale=alt.Scale(domain=[-8, 5])),
        detail='scene:N',
        color=alt.value(col)
    )

    # Mean line
    mean_line = alt.Chart(mc).mark_line(strokeWidth=2.5, point=True).encode(
        x='value:Q',
        y=alt.Y('mean:Q', scale=alt.Scale(domain=[-8, 5])),
        color=alt.value(col)
    )

    # CI band
    band = alt.Chart(mc).mark_errorband(opacity=0.2).encode(
        x='value:Q',
        y=alt.Y('mean:Q', scale=alt.Scale(domain=[-8, 5])),
        yError='ci95:Q',
        color=alt.value(col)
    )

    charts.append((individual + band + mean_line).properties(
        title=f'{res}', width=220, height=180
    ))

alt.hconcat(*charts).properties(title='Alpha Weight: Per-Scene Curves + Mean (by Resolution)')

alt.HConcatChart(...)

### Alpha weight: peak value distribution per resolution

In [17]:
peak_hist = alt.Chart(peaks).mark_bar(opacity=0.7).encode(
    x=alt.X('peak_value:Q', bin=alt.Bin(step=0.1), title='Peak alpha_weight value'),
    y=alt.Y('count():Q', title='Number of scenes'),
    color=alt.Color('resolution:N', scale=color_scale, sort=res_order),
    column=alt.Column('resolution:N', sort=res_order)
).properties(width=150, height=150, title='Distribution of Optimal alpha_weight Per Scene')

peak_hist

alt.Chart(...)

In [18]:
from numpy.polynomial import polynomial as P

quad_records = []

for (res), group in alpha_df.groupby('resolution'):
    # Aggregate to mean per value across scenes
    curve = group.groupby('value')['score_centered'].mean()
    x = curve.index.values
    y = curve.values

    # Fit quadratic
    coeffs = np.polyfit(x, y, 2)  # [a, b, c] for ax² + bx + c
    a, b, c = coeffs
    peak_x = -b / (2 * a)  # vertex of parabola

    # R² 
    y_pred = np.polyval(coeffs, x)
    ss_res = ((y - y_pred) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot

    quad_records.append({
        'resolution': res,
        'a (quadratic)': round(a, 4),
        'b (linear)': round(b, 4),
        'c (intercept)': round(c, 4),
        'peak_at': round(peak_x, 3),
        'R²': round(r2, 4),
        'concave_down': a < 0
    })

    # Plot fitted curve
    x_smooth = np.linspace(x.min(), x.max(), 100)
    y_smooth = np.polyval(coeffs, x_smooth)
    fit_df = pd.DataFrame({'value': x_smooth, 'fitted': y_smooth, 'resolution': res})
    quad_records_plot = fit_df if res == res_order[0] else pd.concat([quad_records_plot, fit_df])
    if res == res_order[0]:
        quad_records_plot = fit_df
    else:
        quad_records_plot = pd.concat([quad_records_plot, fit_df])

quad_summary = pd.DataFrame(quad_records)
print(quad_summary.to_string(index=False))

resolution  a (quadratic)  b (linear)  c (intercept)  peak_at     R²  concave_down
      100%        -6.8122      7.9505        -1.2195    0.584 0.9255          True
       50%       -10.1576      4.9275         1.0277    0.243 0.9501          True
       71%        -9.4751      7.9869        -0.4342    0.421 0.9023          True
       87%        -7.6278      8.1175        -1.0430    0.532 0.9205          True


###  hist_percent: per-scene Spearman r + range

In [19]:
hist_df = df[df['parameter'] == 'hist_percent'].copy()

spearman_records = []
range_records = []

for (scene, res), group in hist_df.groupby(['scene', 'resolution']):
    val_means = group.groupby('value')['score_centered'].mean()
    if len(val_means) < 3:
        continue
    r, p = stats.spearmanr(val_means.index, val_means.values)
    score_range = group.groupby('value')['score'].mean().max() - group.groupby('value')['score'].mean().min()
    spearman_records.append({'scene': scene, 'resolution': res, 'spearman_r': r, 'p_value': p})
    range_records.append({'scene': scene, 'resolution': res, 'score_range': score_range})

spearman_hist = pd.DataFrame(spearman_records)
range_hist = pd.DataFrame(range_records)
spearman_hist['resolution'] = spearman_hist['resolution'].astype(str) + '%'
range_hist['resolution'] = range_hist['resolution'].astype(str) + '%'

# Spearman r distribution
r_plot = alt.Chart(spearman_hist).mark_bar(opacity=0.7).encode(
    x=alt.X('spearman_r:Q', bin=alt.Bin(step=0.2), title='Spearman r'),
    y=alt.Y('count():Q', title='Scenes'),
    color=alt.Color('resolution:N', scale=color_scale, sort=res_order),
    column=alt.Column('resolution:N', sort=res_order)
).properties(width=130, height=150, title='hist_percent: Per-Scene Spearman r (monotonic consistency)')

# Score range distribution
range_plot = alt.Chart(range_hist).mark_bar(opacity=0.7).encode(
    x=alt.X('score_range:Q', bin=alt.Bin(step=0.5), title='Score range (pts)'),
    y=alt.Y('count():Q', title='Scenes'),
    color=alt.Color('resolution:N', scale=color_scale, sort=res_order),
    column=alt.Column('resolution:N', sort=res_order)
).properties(width=130, height=150, title='hist_percent: Per-Scene Score Range vs 5pt threshold')

# Reference line at 5 pts
threshold_line = alt.Chart(pd.DataFrame({'x': [5]})).mark_rule(
    color='red', strokeDash=[4,4]
).encode(x='x:Q')

r_plot & (range_plot + threshold_line)

TypeError: Faceted charts cannot be layered. Instead, layer the charts before faceting.

In [20]:
from scipy.stats import ttest_1samp

EQUIVALENCE_BOUND = 2.0  # points — half the 5pt perceptual threshold, conservative

tost_records = []

for param in ['filter_size', 'num_samples']:
    for (res), group in df[df['parameter'] == param].groupby('resolution'):
        # Per scene: compute range across parameter values
        scene_ranges = group.groupby('scene').apply(
            lambda s: s.groupby('value')['score'].mean().max() - s.groupby('value')['score'].mean().min()
        )
        # TOST: test that mean range is less than EQUIVALENCE_BOUND
        # H0: effect >= bound  →  reject = effect is negligible
        t_upper, p_upper = ttest_1samp(scene_ranges, EQUIVALENCE_BOUND)
        t_lower, p_lower = ttest_1samp(scene_ranges, -EQUIVALENCE_BOUND)
        tost_p = max(p_upper / 2, p_lower / 2)  # one-tailed
        tost_records.append({
            'parameter': param,
            'resolution': res,
            'mean_range': round(scene_ranges.mean(), 4),
            'std_range': round(scene_ranges.std(), 4),
            'TOST_p': round(tost_p, 6),
            'equivalent': tost_p < 0.05,
            'bound': EQUIVALENCE_BOUND
        })

tost_df = pd.DataFrame(tost_records)
print(tost_df.to_string(index=False))

  parameter  resolution  mean_range  std_range  TOST_p  equivalent  bound
filter_size          50      0.1317     0.1741     0.0        True    2.0
filter_size          71      0.1354     0.2231     0.0        True    2.0
filter_size          87      0.1530     0.2399     0.0        True    2.0
filter_size         100      0.1581     0.2483     0.0        True    2.0
num_samples          50      0.1599     0.1950     0.0        True    2.0
num_samples          71      0.1574     0.2195     0.0        True    2.0
num_samples          87      0.1769     0.2410     0.0        True    2.0
num_samples         100      0.2926     0.2565     0.0        True    2.0


/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_47257/4105432956.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scene_ranges = group.groupby('scene').apply(
/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_47257/4105432956.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scene_ranges = group.groupby('scene').apply(
/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_47257/4105432

In [21]:
flat_params = ['filter_size', 'num_samples']
agg_flat = df[df['parameter'].isin(flat_params)].groupby(
    ['parameter', 'resolution', 'value'])['score_centered'].agg(['mean','std','count']).reset_index()
agg_flat['ci95'] = 1.96 * agg_flat['std'] / np.sqrt(agg_flat['count'])
agg_flat['resolution'] = agg_flat['resolution'].astype(str) + '%'

# Equivalence band (±2 pts)
band_df = pd.DataFrame({'y': [EQUIVALENCE_BOUND], 'y2': [-EQUIVALENCE_BOUND]})
eq_band = alt.Chart(band_df).mark_rect(opacity=0.08, color='gray').encode(
    y='y2:Q', y2='y:Q'
)
eq_upper = alt.Chart(band_df).mark_rule(strokeDash=[4,4], color='gray', opacity=0.5).encode(y='y:Q')
eq_lower = alt.Chart(band_df).mark_rule(strokeDash=[4,4], color='gray', opacity=0.5).encode(y='y2:Q')

charts = []
for param in flat_params:
    sub = agg_flat[agg_flat['parameter'] == param]
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Centered CGVQM', scale=alt.Scale(domain=[-3, 3])),
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    band = alt.Chart(sub).mark_errorband(opacity=0.15).encode(
        x='value:Q',
        y=alt.Y('mean:Q', scale=alt.Scale(domain=[-3, 3])),
        yError='ci95:Q',
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    charts.append((eq_band + eq_upper + eq_lower + band + line).properties(
        title=param, width=300, height=220
    ))

alt.hconcat(*charts).resolve_scale(color='shared').properties(
    title=f'filter_size & num_samples: Effects Within ±{EQUIVALENCE_BOUND}pt Equivalence Band'
)

alt.HConcatChart(...)

### Statistical Tests 
The issue with using some of these tests is that they want some sort of monotonic increase--this is not what happens with alpha weight. So, they often show significance weirdly...

#### Anova Test (said everything was insignificant)

In [8]:
# Anova shows everything as insignificant...
anova_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    groups_by_value = [g['score'].values for _, g in group.groupby('value')]
    if len(groups_by_value) < 2:
        continue
    f_stat, p_val = stats.f_oneway(*groups_by_value)
    anova_records.append({
        'parameter': param,
        'resolution': res,
        'F': round(f_stat, 4),
        'p_value': round(p_val, 6),
        'significant': p_val < 0.05,
    })

anova_df = pd.DataFrame(anova_records)
anova_df

,parameter,resolution,F,p_value,significant
0,alpha_weight,50,0.9945,0.451983,False
1,alpha_weight,71,0.4638,0.924072,False
2,alpha_weight,87,0.5460,0.870472,False
3,alpha_weight,100,0.8543,0.585960,False
4,filter_size,50,0.0502,0.999473,False
5,filter_size,71,0.0000,1.000000,False
6,filter_size,87,0.0000,1.000000,False
7,filter_size,100,0.0029,1.000000,False
8,hist_percent,50,0.0234,0.995114,False
9,hist_percent,71,0.1846,0.906569,False


In [9]:
effect_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    scene_ranges = group.groupby('scene').apply(
        lambda s: s.groupby('value')['score'].mean().max() - s.groupby('value')['score'].mean().min()
    )
    effect_records.append({
        'parameter': param,
        'resolution': res,
        'mean_range': scene_ranges.mean(),
        'se_range': scene_ranges.sem()
    })

effect_df = pd.DataFrame(effect_records)
effect_df['range_label'] = effect_df['mean_range'].apply(lambda x: f'{x:.2f}')

/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_47257/1139419750.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scene_ranges = group.groupby('scene').apply(
/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_47257/1139419750.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scene_ranges = group.groupby('scene').apply(
/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_47257/113941975

In [10]:
from scipy.stats import friedmanchisquare

# ── Friedman ──────────────────────────────────────────────────────────────────
def compute_friedman(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        value_counts = group.groupby('scene')['value'].nunique()
        n_values = group['value'].nunique()
        complete_scenes = value_counts[value_counts == n_values].index
        group = group[group['scene'].isin(complete_scenes)]
        if group['scene'].nunique() < 3:
            continue
        values = sorted(group['value'].unique())
        groups_by_value = [group[group['value'] == v]['score_centered'].values for v in values]
        if any(len(g) < 3 for g in groups_by_value):
            continue
        _, p = friedmanchisquare(*groups_by_value)
        records.append({'parameter': param, 'resolution': res, 'p_value': p, 'test': 'Friedman'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

# ── Kendall's W ───────────────────────────────────────────────────────────────
def kendalls_w(data):
    n, k = data.shape
    ranks = data.rank(axis=1)
    rank_sums = ranks.sum(axis=0)
    S = ((rank_sums - rank_sums.mean()) ** 2).sum()
    W = 12 * S / (n ** 2 * (k ** 3 - k))
    chi2 = n * (k - 1) * W
    p = 1 - stats.chi2.cdf(chi2, df=k - 1)
    return W, p

def compute_kendall(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        value_counts = group.groupby('scene')['value'].nunique()
        n_values = group['value'].nunique()
        complete_scenes = value_counts[value_counts == n_values].index
        group = group[group['scene'].isin(complete_scenes)]
        if group['scene'].nunique() < 3:
            continue
        pivot = group.pivot_table(index='scene', columns='value', values='score_centered').dropna()
        if pivot.shape[0] < 3 or pivot.shape[1] < 2:
            continue
        W, p = kendalls_w(pivot)
        records.append({'parameter': param, 'resolution': res, 'p_value': p, 'W': round(W, 4), 'test': 'Kendall W'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

# ── Spearman ──────────────────────────────────────────────────────────────────
def compute_spearman(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        scene_corrs = []
        for scene, sgroup in group.groupby('scene'):
            val_means = sgroup.groupby('value')['score_centered'].mean()
            if len(val_means) < 3:
                continue
            r, _ = stats.spearmanr(val_means.index, val_means.values)
            scene_corrs.append(r)
        if len(scene_corrs) < 3:
            continue
        t, p = stats.ttest_1samp(scene_corrs, 0)
        records.append({'parameter': param, 'resolution': res, 'mean_r': round(np.mean(scene_corrs), 4), 'p_value': round(p, 6), 'test': 'Spearman'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

In [11]:
# ── Compute all ───────────────────────────────────────────────────────────────
sig_friedman = compute_friedman(df)
sig_kendall  = compute_kendall(df)
sig_spearman = compute_spearman(df)

In [12]:
# ── Toggle this to switch tests in all downstream plots ───────────────────────
sig_df = sig_spearman   # ← change to sig_friedman, sig_kendall, or sig_spearman
print(sig_df)

       parameter  resolution  mean_r   p_value      test  significant
0   alpha_weight          50 -0.4729  0.000002  Spearman         True
1   alpha_weight          71  0.0077  0.943799  Spearman        False
2   alpha_weight          87  0.3760  0.001073  Spearman         True
3   alpha_weight         100  0.5375  0.000001  Spearman         True
4    filter_size          50  0.0876  0.262613  Spearman        False
5    filter_size          71  0.1332  0.074959  Spearman        False
6    filter_size          87  0.0545  0.445981  Spearman        False
7    filter_size         100  0.0603  0.405259  Spearman        False
8   hist_percent          50  0.0286  0.842145  Spearman        False
9   hist_percent          71  0.8667  0.000000  Spearman         True
10  hist_percent          87  0.9333  0.000000  Spearman         True
11  hist_percent         100  0.9619  0.000000  Spearman         True
12   num_samples          50  0.0857  0.369821  Spearman        False
13   num_samples    

In [13]:
summary_df = effect_df.merge(sig_df[['parameter', 'resolution', 'p_value', 'significant', 'test']], on=['parameter', 'resolution'])

base = alt.Chart(summary_df)

heatmap = base.mark_rect().encode(
    x=alt.X('resolution:O', title='Base Resolution', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N', title='TAA Parameter'),
    color=alt.Color('mean_range:Q',
                    scale=alt.Scale(scheme='blues', domain=[0, 5]),
                    title='Mean Score Range (pts)'),
    tooltip=['parameter', 'resolution',
             alt.Tooltip('mean_range:Q', format='.3f'),
             alt.Tooltip('p_value:Q', format='.4f'),
             'significant', 'test']
)

range_text = base.mark_text(fontSize=11, dy=-6).encode(
    x=alt.X('resolution:O', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N'),
    text='range_label:N',
    color=alt.condition(alt.datum.mean_range > 2.5, alt.value('white'), alt.value('black'))
)

sig_text = base.mark_text(fontSize=14, dy=6).encode(
    x=alt.X('resolution:O', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N'),
    text=alt.condition(alt.datum.significant, alt.value('✱'), alt.value('')),
    color=alt.condition(alt.datum.mean_range > 2.5, alt.value('white'), alt.value('black'))
)

(heatmap + range_text + sig_text).properties(
    width=300, height=200,
    title=f'TAA Parameter Effect × Resolution ({summary_df["test"].iloc[0]})'
)

alt.LayerChart(...)

In [14]:
# Do scenes agree on the direction of the effect for each parameter?
for param in PARAMS:
    sub = df[df['parameter'] == param]
    scene_corrs = []
    for scene, sgroup in sub.groupby('scene'):
        val_means = sgroup.groupby('value')['score_centered'].mean()
        if len(val_means) < 3:
            continue
        r, _ = stats.spearmanr(val_means.index, val_means.values)
        scene_corrs.append(r)
    print(f"{param}: mean_r={np.mean(scene_corrs):.3f}, std_r={np.std(scene_corrs):.3f}, n={len(scene_corrs)}")

alpha_weight: mean_r=0.022, std_r=0.484, n=21
hist_percent: mean_r=0.705, std_r=0.477, n=21
filter_size: mean_r=0.127, std_r=0.368, n=21
num_samples: mean_r=0.410, std_r=0.516, n=21
